## Data Foundation

In [1]:
import pandas as pd
import os
from pathlib import Path

# print(os.getcwd())
customers = pd.read_csv('../data/raw/customers.csv')
products = pd.read_csv('../data/raw/products.csv')
orders = pd.read_csv('../data/raw/orders.csv')
tickets = pd.read_csv('../data/raw/tickets.csv',keep_default_na=False)

print("\n========== RAW DATASETS LOADED ==========")

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Tickets:", tickets.shape)



========== RAW DATASETS LOADED ==========
Customers: (1200, 5)
Products: (40, 4)
Orders: (13084, 9)
Tickets: (164, 5)


In [2]:
print("\n========== DATASET Dimensions ==========")

datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders,
    "Tickets": tickets
}

for name, df in datasets.items():
    rows, cols = df.shape
    print(f"\n{name}")
    print(f"Rows: {rows}")
    print(f"Columns: {cols}")


========== DATASET Dimensions ==========

Customers
Rows: 1200
Columns: 5

Products
Rows: 40
Columns: 4

Orders
Rows: 13084
Columns: 9

Tickets
Rows: 164
Columns: 5


In [3]:
print("\n========== DATA TYPES ==========")

for name, df in datasets.items():

    print(f"\n{name} - Data Types")

    print(df.dtypes)


========== DATA TYPES ==========

Customers - Data Types
customer_id        str
city               str
signup_date        str
age            float64
gender             str
dtype: object

Products - Data Types
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object

Orders - Data Types
order_id                str
customer_id             str
product_id              str
quantity              int64
order_ts                str
status                  str
discount_pct          int64
order_amount            str
delivery_minutes    float64
dtype: object

Tickets - Data Types
ticket_id        str
customer_id      str
order_id         str
text             str
true_category    str
dtype: object


In [4]:
print("\n========== MISSING VALUE AUDIT ==========")

for name, df in datasets.items():

    print(f"\n{name} - Missing Values")

    missing_count = df.isna().sum()

    missing_pct = df.isna().mean() * 100

    missing_report = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct.round(2)
    })

    print(missing_report)



========== MISSING VALUE AUDIT ==========

Customers - Missing Values
             missing_count  missing_pct
customer_id              0         0.00
city                     0         0.00
signup_date              0         0.00
age                     61         5.08
gender                  30         2.50

Products - Missing Values
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

Orders - Missing Values
                  missing_count  missing_pct
order_id                      0         0.00
customer_id                   0         0.00
product_id                    0         0.00
quantity                      0         0.00
order_ts                      0         0.00
status                        0         0.00
discount_pct                  0         0.00
order_amount                 53         0.41
delivery_minutes           

In [5]:
print("\n========== CUSTOMER-SPECIFIC AUDIT ==========")

age_numeric = pd.to_numeric(
    customers["age"],
    errors="coerce"
)

age_text = customers["age"].astype("string").str.strip()

blank_ages = age_text.isna() | age_text.eq("")

unparseable_ages = (
    ~blank_ages &
    age_numeric.isna()
)

invalid_age_range = (
    age_numeric.notna() &
    ~age_numeric.between(10, 100)
)

print("Blank ages:", int(blank_ages.sum()))

print("Unparseable ages:", int(unparseable_ages.sum()))

print("Ages outside 10-100:", int(invalid_age_range.sum()))

print(
    "Total blank or invalid ages:",
    int((blank_ages | unparseable_ages | invalid_age_range).sum())
)


========== CUSTOMER-SPECIFIC AUDIT ==========
Blank ages: 61
Unparseable ages: 0
Ages outside 10-100: 6
Total blank or invalid ages: 67


In [6]:
print("\nRaw city spellings:")

print(
    customers["city"]
    .value_counts(dropna=False)
)


Raw city spellings:
city
Pune           148
Delhi          123
Hyderabad      119
Bengaluru      115
Mumbai         113
Chennai        106
chennai         44
hyderabad       40
delhi           36
mumbai          35
pune            35
BENGALURU       32
 Delhi          30
bengaluru       27
PUNE            24
 Bengaluru      23
DELHI           21
CHENNAI         21
 Chennai        20
 Pune           20
 Mumbai         19
 Hyderabad      18
MUMBAI          17
HYDERABAD       14
Name: count, dtype: int64


In [7]:
print("\nRaw gender spellings:")

print(
    customers["gender"]
    .value_counts(dropna=False)
)



Raw gender spellings:
gender
female    136
FEMALE    122
F         120
m         118
f         117
Female    115
male      114
MALE      110
M         110
Male      108
NaN        30
Name: count, dtype: int64


In [8]:
print("\n========== ORDER-SPECIFIC AUDIT ==========")

exact_duplicates = orders.duplicated().sum()

print("Exact duplicate rows:", int(exact_duplicates))

pending_records = (
    orders["status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("pending")
    .sum()
)

print("Pending records:", int(pending_records))

distinct_order_ids = orders["order_id"].nunique()

print("Distinct order IDs:", int(distinct_order_ids))
amount_text = orders["order_amount"].astype("string").str.strip()

amount_numeric = pd.to_numeric(
    orders["order_amount"],
    errors="coerce"
)

blank_amounts = (
    amount_text.isna() |
    amount_text.eq("")
)

unparseable_amounts = (
    ~blank_amounts &
    amount_numeric.isna()
)

print("Blank order amounts:", int(blank_amounts.sum()))

print(
    "Unparseable order amounts:",
    int(unparseable_amounts.sum())
)

delivery_text = (
    orders["delivery_minutes"]
    .astype("string")
    .str.strip()
)

delivery_numeric = pd.to_numeric(
    orders["delivery_minutes"],
    errors="coerce"
)

blank_delivery = (
    delivery_text.isna() |
    delivery_text.eq("")
)

unparseable_delivery = (
    ~blank_delivery &
    delivery_numeric.isna()
)

negative_delivery = (
    delivery_numeric.notna() &
    delivery_numeric.lt(0)
)

print("Blank delivery times:", int(blank_delivery.sum()))

print(
    "Unparseable delivery times:",
    int(unparseable_delivery.sum())
)

print(
    "Negative delivery times:",
    int(negative_delivery.sum())
)



========== ORDER-SPECIFIC AUDIT ==========
Exact duplicate rows: 119
Pending records: 656
Distinct order IDs: 12309
Blank order amounts: 53
Unparseable order amounts: 67
Blank delivery times: 2541
Unparseable delivery times: 0
Negative delivery times: 106


In [9]:
customers_clean = customers.copy()

customers_clean["city"] = (customers_clean["city"].astype("string")
    .str.strip())

customers_clean["city"] = (customers_clean["city"].str.title())

valid_cities = ["Chennai", "Bengaluru", "Mumbai", "Delhi", "Hyderabad", "Pune"]

unexpected_cities = (customers_clean.loc[~customers_clean["city"].isin(valid_cities), "city"].drop_duplicates().to_list())

print("\nUnexpected city values:", unexpected_cities)

assert len(unexpected_cities) == 0, (
    f"Unexpected city names found: {unexpected_cities}"
)

print("Unique cities:", customers_clean["city"].nunique())

print(customers_clean["city"].value_counts())


Unexpected city values: []
Unique cities: 6
city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: Int64


In [10]:
gender_normalized = customers_clean["gender"].astype("string").str.strip().str.lower()

gender_mapping = {
    "m": "male",
    "male": "male",
    "f": "female",
    "female": "female","": "unknown"}

unexpected_genders = (gender_normalized[gender_normalized.notna()& ~gender_normalized.isin(gender_mapping.keys())].drop_duplicates().to_list())

print("\nUnexpected gender values:", unexpected_genders)

customers_clean["gender"] = (gender_normalized.map(gender_mapping).fillna("unknown"))

print(customers_clean["gender"].value_counts())




Unexpected gender values: []
gender
female     610
male       560
unknown     30
Name: count, dtype: int64


In [11]:
customers_clean["age"] = pd.to_numeric(
    customers_clean["age"],
    errors="coerce"
)

print(customers_clean["age"].dtype)

invalid_age_mask = ~customers_clean["age"].between(10, 100)

invalid_age_count = int(invalid_age_mask.sum())

print("Invalid age count:", invalid_age_count)

customers_clean.loc[invalid_age_mask, "age"] = float("nan")

city_median = customers_clean.groupby("city")["age"].transform("median")

customers_clean["age"] = customers_clean["age"].fillna(city_median)

print("Missing ages after imputation:", int(customers_clean["age"].isna().sum()))

assert customers_clean["age"].between(10, 100).all(), (
    "Some ages are still missing or outside the permitted range."
)

print("All customer ages are valid.")

float64
Invalid age count: 67
Missing ages after imputation: 0
All customer ages are valid.


In [12]:
customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"], errors="raise")
print(customers_clean["signup_date"].dtype)

datetime64[us]


In [13]:
print("Total Customers:",len(customers_clean))
print("number of unique cities:", customers_clean["city"].nunique())
print("\n Gender distribution:")
print(customers_clean["gender"].value_counts().reindex(["female","male","unknown"], fill_value=0))

print("\n missing ages:",customers_clean["age"].isna().sum())

print("Age range:",customers_clean["age"].min(),"-",customers_clean["age"].max())

print("\n signup date data type:",customers_clean["signup_date"].dtype)

assert len(customers_clean) == 1200

assert customers_clean["city"].nunique() == 6

assert customers_clean["gender"].value_counts().to_dict() == {"female": 610, "male": 560, "unknown": 30}

assert customers_clean["age"].notna().all()

assert customers_clean["age"].between(10, 100).all()

assert pd.api.types.is_datetime64_any_dtype(customers_clean["signup_date"])

print("\n customers cleaning checkpoints passed")

Total Customers: 1200
number of unique cities: 6

 Gender distribution:
gender
female     610
male       560
unknown     30
Name: count, dtype: int64

 missing ages: 0
Age range: 18.0 - 60.0

 signup date data type: datetime64[us]

 customers cleaning checkpoints passed


In [14]:
orders_clean = orders.copy()

orders_clean["order_ts"] = pd.to_datetime(orders_clean["order_ts"], errors="raise")

print("Order timestamp type:", orders_clean["order_ts"].dtype)

Order timestamp type: datetime64[us]


In [15]:
duplicates_before = orders_clean.duplicated().sum()

print("Exact duplicate rows before cleaning:", int(duplicates_before))

orders_clean = orders_clean.drop_duplicates().copy()

duplicates_after = orders_clean.duplicated().sum()

print("Exact duplicate rows after cleaning:", int(duplicates_after))

assert duplicates_after == 0

orders_clean = orders_clean.sort_values(by="order_ts", ascending=True).copy()

orders_clean = orders_clean.drop_duplicates(subset=["order_id"], keep="last").copy()

orders_clean = orders_clean.reset_index(drop=True)

print("Distinct order IDs after cleaning:", int(orders_clean["order_id"].nunique()))

print("remaining duplicate order ids:", orders_clean["order_id"].duplicated().sum())

Exact duplicate rows before cleaning: 119
Exact duplicate rows after cleaning: 0
Distinct order IDs after cleaning: 12309
remaining duplicate order ids: 0


In [16]:
product_details = products[["product_id", "category", "price_inr"]].copy()

product_details["price_inr"] = pd.to_numeric(product_details["price_inr"], errors="raise")

orders_clean = orders_clean.merge(product_details, on="product_id", how="left", validate = "many_to_one")

assert orders_clean["category"].notna().all(),("some orders do not have a matching product.")

assert orders_clean["price_inr"].notna().all(),("some orders do not have a matching product price.")
print("Orders with product information:", len(orders_clean))

Orders with product information: 12309


In [17]:
orders_clean["order_amount"] = pd.to_numeric(orders_clean["order_amount"], errors="coerce")

print("Order amount data type:", orders_clean["order_amount"].dtype)

print("Missing amounts after conversion:", orders_clean["order_amount"].isna().sum())

missing_amount_mask = orders_clean["order_amount"].isna()
missing_amount_count = int(missing_amount_mask.sum())
print("\nMissing amounts before calculation:", missing_amount_count)

calculated_amount = orders_clean["price_inr"] * orders_clean["quantity"] * (1 - orders_clean["discount_pct"] / 100).round(2)

orders_clean.loc[missing_amount_mask, "order_amount"] = calculated_amount[missing_amount_mask]

print("Missing amounts after calculation:", orders_clean["order_amount"].isna().sum())

assert orders_clean["order_amount"].notna().all(),("some orders still have missing order amounts.")

print("All orders have valid order amounts.")

Order amount data type: float64
Missing amounts after conversion: 112

Missing amounts before calculation: 112
Missing amounts after calculation: 0
All orders have valid order amounts.


In [18]:
orders_clean["delivery_minutes"] = pd.to_numeric(orders_clean["delivery_minutes"], errors="coerce")

negative_delivery_mask = orders_clean["delivery_minutes"] < 0

negative_delivery_count = int(negative_delivery_mask.sum())

print("\n Negative delivery times found:", negative_delivery_count)

orders_clean.loc[negative_delivery_mask, "delivery_minutes"] = float("nan")

print("Negative delivery times replaced with missing values.")


 Negative delivery times found: 105
Negative delivery times replaced with missing values.


In [19]:
completed_mask = orders_clean["status"].eq("completed")

category_median = (orders_clean.loc[completed_mask].groupby("category")["delivery_minutes"].transform("median"))

orders_clean.loc[completed_mask, "delivery_minutes"] = (orders_clean.loc[completed_mask, "delivery_minutes"].fillna(category_median))


In [20]:
completed_missing_delivery = (
    orders_clean.loc[
        orders_clean["status"].eq("completed"),
        "delivery_minutes"
    ].isna().sum()
)

cancelled_missing_delivery = (
    orders_clean.loc[
        orders_clean["status"].eq("cancelled"),
        "delivery_minutes"
    ].isna().sum()
)

print(
    "\nMissing delivery times in completed orders:",
    completed_missing_delivery
)

print(
    "Missing delivery times in cancelled orders:",
    cancelled_missing_delivery
)


Missing delivery times in completed orders: 0
Missing delivery times in cancelled orders: 1199


In [21]:
print("\nOrder status distribution:")

print(orders_clean["status"].value_counts())

print("\n unique orders:",orders_clean["order_id"].nunique())

completed_count = int(orders_clean["status"].eq("completed").sum())
cancelled_count = int(orders_clean["status"].eq("cancelled").sum())
pending_count = int(orders_clean["status"].eq("pending").sum())

print("\nCompleted orders:", completed_count)

print("Cancelled orders:", cancelled_count)

print("Pending orders:", pending_count)

missing_amount_count = int(orders_clean["order_amount"].isna().sum())

print("\nMissing order amounts:", missing_amount_count)

completed_missing = int(
    orders_clean.loc[
        orders_clean["status"].eq("completed"),
        "delivery_minutes"
    ].isna().sum()
)

print("Missing completed delivery times:", completed_missing)

cancelled_missing = int(
    orders_clean.loc[
        orders_clean["status"].eq("cancelled"),
        "delivery_minutes"
    ].isna().sum()
)

print("Missing cancelled delivery times:",cancelled_missing)

completed_revenue = orders_clean.loc[
    orders_clean["status"].eq("completed"),
    "order_amount"
].sum()

print("\nCompleted-order revenue: INR",f"{completed_revenue:,.2f}")



Order status distribution:
status
completed    11110
cancelled     1199
Name: count, dtype: int64

 unique orders: 12309

Completed orders: 11110
Cancelled orders: 1199
Pending orders: 0

Missing order amounts: 0
Missing completed delivery times: 0
Missing cancelled delivery times: 1199

Completed-order revenue: INR 2,998,668.80


In [22]:
assert len(orders_clean) == 12309

assert orders_clean["order_id"].nunique() == 12309

assert orders_clean["order_id"].duplicated().sum() == 0

assert completed_count == 11110

assert cancelled_count == 1199

assert pending_count == 0

assert missing_amount_count == 0

assert completed_missing == 0

assert cancelled_missing == 1199

assert round(completed_revenue, 2) == 2998668.80


In [23]:
path = Path('../data/clean')
print(path)

path.mkdir(parents=True, exist_ok=True)
customers_clean.to_csv(path/'customers_clean.csv', index=False)

orders_clean["order_ts"] = pd.to_datetime(orders_clean["order_ts"], errors="coerce")
orders_clean["order_ts"] = orders_clean["order_ts"].dt.strftime('%Y-%m-%d %H:%M:%S')
orders_clean.to_csv(path/'orders_clean.csv', index=False)
products.to_csv(path/'products_clean.csv', index=False)

print("\n all 3 dataframes saved as csv files successfully in the clean folder")


..\data\clean

 all 3 dataframes saved as csv files successfully in the clean folder


In [24]:
# Check whether the files exist
print(list(path.iterdir()))

[WindowsPath('../data/clean/customers_clean.csv'), WindowsPath('../data/clean/orders_clean.csv'), WindowsPath('../data/clean/products_clean.csv')]


In [25]:
customers_check = pd.read_csv("../data/clean/customers_clean.csv")
products_check = pd.read_csv("../data/clean/products_clean.csv")
orders_check = pd.read_csv("../data/clean/orders_clean.csv")

print(customers_check.shape)
print(products_check.shape)
print(orders_check.shape)

(1200, 5)
(40, 4)
(12309, 11)


In [26]:
assert customers_check.shape == customers_clean.shape
assert products_check.shape == products.shape
assert orders_check.shape == orders_clean.shape

print("CSV validation successful!")

CSV validation successful!


In [73]:

import sqlite3
from pathlib import Path

# Step 1: Define the database path
db_path = Path("../data/instamart.db")

# Step 2: Ensure the parent directory exists
db_path.parent.mkdir(parents=True, exist_ok=True)

# Step 3: Establish the SQLite connection
conn = sqlite3.connect(db_path)

try:

    # Step 4: Insert customers DataFrame
    customers_clean.to_sql(
        "customers",
        conn,
        if_exists="replace",
        index=False
    )

    # Step 5: Insert products DataFrame
    products.to_sql(
        "products",
        conn,
        if_exists="replace",
        index=False
    )

    # Step 6: Insert orders DataFrame
    orders_clean.to_sql(
        "orders",
        conn,
        if_exists="replace",
        index=False
    )

    # Step 7: Commit the changes
    conn.commit()

    print("All three tables successfully created!")

finally:

    # Step 8: Close the connection
    conn.close()


All three tables successfully created!


In [74]:
conn = sqlite3.connect("../data/instamart.db")

In [75]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

tables = pd.read_sql_query(query, conn)

print(tables)

        name
0  customers
1   products
2     orders


In [76]:
customer_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM customers",
    conn
)

product_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM products",
    conn
)

order_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM orders",
    conn
)

print("Customers:", customer_count.iloc[0, 0])
print("Products:", product_count.iloc[0, 0])
print("Orders:", order_count.iloc[0, 0])

Customers: 1200
Products: 40
Orders: 12309


In [78]:
assert customer_count.iloc[0, 0] == len(customers_clean)

assert product_count.iloc[0, 0] == len(products)

assert order_count.iloc[0, 0] == len(orders_clean)

print("Row count validation successful!")

Row count validation successful!


In [79]:
for table in ["customers", "products", "orders"]:

    print(f"\nSchema of {table}:")

    schema = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    )

    print(schema[["name", "type"]])


Schema of customers:
          name       type
0  customer_id       TEXT
1         city       TEXT
2  signup_date  TIMESTAMP
3          age       REAL
4       gender       TEXT

Schema of products:
           name     type
0    product_id     TEXT
1  product_name     TEXT
2      category     TEXT
3     price_inr  INTEGER

Schema of orders:
                name     type
0           order_id     TEXT
1        customer_id     TEXT
2         product_id     TEXT
3           quantity  INTEGER
4           order_ts     TEXT
5             status     TEXT
6       discount_pct  INTEGER
7       order_amount     REAL
8   delivery_minutes     REAL
9           category     TEXT
10         price_inr  INTEGER


In [80]:
sample_orders = pd.read_sql_query(
    """
    SELECT order_id, order_ts
    FROM orders
    LIMIT 5
    """,
    conn
)

print(sample_orders)

     order_id             order_ts
0  ORD-110093  2025-04-01 07:08:28
1  ORD-111396  2025-04-01 07:11:59
2  ORD-102006  2025-04-01 07:12:43
3  ORD-105921  2025-04-01 07:54:43
4  ORD-108115  2025-04-01 08:02:58


In [81]:
timestamp_type = pd.read_sql_query(
    """
    SELECT typeof(order_ts) AS storage_type
    FROM orders
    LIMIT 5
    """,
    conn
)

print(timestamp_type)

  storage_type
0         text
1         text
2         text
3         text
4         text


In [82]:
invalid_timestamps = pd.read_sql_query(
    """
    SELECT COUNT(*) AS invalid_count
    FROM orders
    WHERE order_ts IS NULL
       OR strftime('%Y-%m-%d %H:%M:%S', order_ts) IS NULL
       OR strftime('%Y-%m-%d %H:%M:%S', order_ts) != order_ts
    """,
    conn
)

print(invalid_timestamps)

   invalid_count
0              0


In [83]:
conn.close()

In [84]:


# Reopen the database
conn = sqlite3.connect('../data/instamart.db')

try:

    # 1. Verify table names
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        """,
        conn
    )

    expected_tables = {"customers", "products", "orders"}
    actual_tables = set(tables["name"])

    assert expected_tables.issubset(actual_tables)

    print("All three tables exist!")

    # 2. Validate row counts
    expected_counts = {
        "customers": len(customers_clean),
        "products": len(products),
        "orders": len(orders_clean)
    }

    for table, expected_count in expected_counts.items():

        result = pd.read_sql_query(
            f"SELECT COUNT(*) AS total FROM {table}",
            conn
        )

        actual_count = result.iloc[0, 0]

        assert actual_count == expected_count

        print(f"{table}: {actual_count} records verified")

    # 3. Validate order timestamp storage
    timestamp_check = pd.read_sql_query(
        """
        SELECT COUNT(*) AS invalid_count
        FROM orders
        WHERE order_ts IS NULL
           OR typeof(order_ts) != 'text'
           OR strftime('%Y-%m-%d %H:%M:%S', order_ts) IS NULL
           OR strftime('%Y-%m-%d %H:%M:%S', order_ts) != order_ts
        """,
        conn
    )

    assert timestamp_check.iloc[0, 0] == 0

    print("Order timestamps verified!")

    print("\nSTEP 5.5 VALIDATION SUCCESSFUL!")

finally:

    conn.close()


All three tables exist!
customers: 1200 records verified
products: 40 records verified
orders: 12309 records verified
Order timestamps verified!

STEP 5.5 VALIDATION SUCCESSFUL!


In [85]:
conn = sqlite3.connect("../data/instamart.db")

tables = pd.read_sql_query(
    """
    SELECT name from sqlite_master where type='table';
    """,conn)

print(tables)

        name
0  customers
1   products
2     orders


In [86]:
customer_schema = pd.read_sql_query(
    "PRAGMA table_info(customers);", conn)

print(customer_schema)

   cid         name       type  notnull dflt_value  pk
0    0  customer_id       TEXT        0       None   0
1    1         city       TEXT        0       None   0
2    2  signup_date  TIMESTAMP        0       None   0
3    3          age       REAL        0       None   0
4    4       gender       TEXT        0       None   0


In [87]:
orders_schema = pd.read_sql_query(
    "PRAGMA table_info(orders);", conn)

print(orders_schema)

    cid              name     type  notnull dflt_value  pk
0     0          order_id     TEXT        0       None   0
1     1       customer_id     TEXT        0       None   0
2     2        product_id     TEXT        0       None   0
3     3          quantity  INTEGER        0       None   0
4     4          order_ts     TEXT        0       None   0
5     5            status     TEXT        0       None   0
6     6      discount_pct  INTEGER        0       None   0
7     7      order_amount     REAL        0       None   0
8     8  delivery_minutes     REAL        0       None   0
9     9          category     TEXT        0       None   0
10   10         price_inr  INTEGER        0       None   0


In [89]:
products_schema = pd.read_sql_query(
    "PRAGMA table_info(products);", conn)

print(products_schema)

   cid          name     type  notnull dflt_value  pk
0    0    product_id     TEXT        0       None   0
1    1  product_name     TEXT        0       None   0
2    2      category     TEXT        0       None   0
3    3     price_inr  INTEGER        0       None   0


In [90]:
for table in ["customers", "products", "orders"]:

    schema = pd.read_sql_query(
        f"PRAGMA table_info({table});",
        conn
    )

    print(f"\n{table.upper()} COLUMNS:")
    print(schema["name"].tolist())


CUSTOMERS COLUMNS:
['customer_id', 'city', 'signup_date', 'age', 'gender']

PRODUCTS COLUMNS:
['product_id', 'product_name', 'category', 'price_inr']

ORDERS COLUMNS:
['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes', 'category', 'price_inr']


In [94]:
conn.close()

In [95]:
from math import isclose

db_path = Path("../data/instamart.db")
sql_dir = Path("../sql")
sql_file = (sql_dir/"queries.sql")

sql_dir.mkdir(parents=True, exist_ok=True)

if not db_path.exists():
    raise FileNotFoundError(f"Database not found: {db_path}")

queries = {

    # Q1: WHERE followed by HAVING
    "Q1": """
    SELECT
        customer_id,
        SUM(order_amount) AS total_spend
    FROM orders
    WHERE status = 'completed'
      AND order_amount > 500
    GROUP BY customer_id
    HAVING SUM(order_amount) > 8000
    ORDER BY total_spend DESC;
    """,

    # Q2: Three-table JOIN
    "Q2": """
    SELECT
        c.city,
        p.category,
        ROUND(SUM(o.order_amount), 2) AS revenue
    FROM orders AS o
    INNER JOIN customers AS c
        ON o.customer_id = c.customer_id
    INNER JOIN products AS p
        ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY c.city, p.category
    ORDER BY c.city, p.category;
    """,

    # Q3: Scalar subquery
    "Q3": """
    SELECT
        product_id,
        category,
        price_inr
    FROM products
    WHERE price_inr > (
        SELECT AVG(price_inr)
        FROM products
    )
    ORDER BY price_inr DESC;
    """,

    # Q4: LEFT JOIN
    "Q4": """
    SELECT
        c.customer_id,
        c.city
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
        AND o.status = 'completed'
    WHERE o.order_id IS NULL
    ORDER BY c.customer_id;
    """,

    # Q5: Common Table Expressions (CTEs)
    "Q5": """
    WITH customer_spend AS (
        SELECT
            customer_id,
            SUM(order_amount) AS total_spend
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    ),

    average_spend AS (
        SELECT
            AVG(total_spend) AS avg_total_spend
        FROM customer_spend
    )

    SELECT
        cs.customer_id,
        ROUND(cs.total_spend, 2) AS total_spend,
        ROUND(av.avg_total_spend, 2) AS average_spend
    FROM customer_spend AS cs
    CROSS JOIN average_spend AS av
    WHERE cs.total_spend > av.avg_total_spend
    ORDER BY cs.total_spend DESC;
    """,

    # Q6: Ranking window functions
    "Q6": """
    WITH customer_spend AS (
        SELECT
            c.customer_id,
            c.city,
            SUM(o.order_amount) AS total_spend
        FROM customers AS c
        INNER JOIN orders AS o
            ON c.customer_id = o.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.customer_id, c.city
    )

    SELECT
        customer_id,
        city,
        ROUND(total_spend, 2) AS total_spend,

        RANK() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS rank_position,

        DENSE_RANK() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS dense_rank_position,

        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS row_number_position

    FROM customer_spend
    ORDER BY city, row_number_position;
    """,

    # Q7: Running total
    "Q7": """
    WITH daily_revenue AS (
        SELECT
            SUBSTR(order_ts, 1, 10) AS day,
            SUM(order_amount) AS revenue
        FROM orders
        WHERE status = 'completed'
        GROUP BY SUBSTR(order_ts, 1, 10)
    )

    SELECT
        day,
        ROUND(revenue, 2) AS revenue,

        ROUND(
            SUM(revenue) OVER (
                ORDER BY day
                ROWS BETWEEN UNBOUNDED PRECEDING
                AND CURRENT ROW
            ),
            2
        ) AS cumulative_revenue

    FROM daily_revenue
    ORDER BY day;
    """
}

with open(sql_file, "w", encoding="utf-8") as file:

    file.write("-- INSTAMART PROJECT: SQL BUSINESS QUERIES\n\n")

    for query_name, query in queries.items():

        file.write(f"\n-- {query_name}\n")
        file.write(query.strip())
        file.write("\n\n")

print(f"SQL queries saved successfully: {sql_file}")

conn = sqlite3.connect(db_path)

try:

    # --------------------------------------------------
    # 5. Execute all seven queries using pandas
    # --------------------------------------------------

    results = {}

    for query_name, query in queries.items():

        print("\n" + "=" * 60)
        print(query_name)
        print("=" * 60)

        result_df = pd.read_sql_query(query, conn)

        results[query_name] = result_df

        print(result_df.to_string(index=False))

        print("\nNumber of rows returned:", len(result_df))


    # --------------------------------------------------
    # 6. Validate results against reference checkpoints
    # --------------------------------------------------

    print("\n\n" + "=" * 60)
    print("REFERENCE CHECKPOINT VALIDATION")
    print("=" * 60)

    # Q1: Expected 30 customers
    q1_count = len(results["Q1"])
    print(f"Q1 customer count: {q1_count}")
    assert q1_count == 30, "Q1 reference check failed"


    # Q2: Expected 30 rows and total revenue INR 2,998,668.80
    q2 = results["Q2"]

    q2_count = len(q2)
    q2_revenue = q2["revenue"].sum()

    print(f"\nQ2 row count: {q2_count}")
    print(f"Q2 total revenue: INR {q2_revenue:,.2f}")

    assert q2_count == 30, "Q2 row count check failed"

    assert isclose(
        q2_revenue,
        2998668.80,
        abs_tol=0.02
    ), "Q2 revenue check failed"

    largest_cell = q2.loc[q2["revenue"].idxmax()]

    print(
        "Q2 largest revenue cell:",
        largest_cell["city"],
        "/",
        largest_cell["category"],
        "/ INR",
        f'{largest_cell["revenue"]:,.2f}'
    )


    # Q3: Expected 11 products
    q3_count = len(results["Q3"])

    print(f"\nQ3 products above average price: {q3_count}")

    assert q3_count == 11, "Q3 reference check failed"


    # Q4: Expected 25 customers
    q4_count = len(results["Q4"])

    print(f"\nQ4 customers with no completed orders: {q4_count}")

    assert q4_count == 25, "Q4 reference check failed"


    # Q5: Expected 550 customers and average INR 2,552.06
    q5 = results["Q5"]

    q5_count = len(q5)
    q5_average = q5["average_spend"].iloc[0] if q5_count else None

    print(f"\nQ5 customers above average spend: {q5_count}")
    print(f"Q5 average completed spend: INR {q5_average:,.2f}")

    assert q5_count == 550, "Q5 customer count check failed"

    assert isclose(
        q5_average,
        2552.06,
        abs_tol=0.01
    ), "Q5 average spend check failed"


    # Q6: Validate top customer per city
    q6 = results["Q6"]

    q6_top = q6[q6["row_number_position"] == 1]

    expected_top_customers = {
        "Bengaluru": "C0082",
        "Chennai": "C0930",
        "Delhi": "C0291",
        "Hyderabad": "C0743",
        "Mumbai": "C0693",
        "Pune": "C0426"
    }

    actual_top_customers = dict(
        zip(q6_top["city"], q6_top["customer_id"])
    )

    print("\nQ6 top customer per city:")
    print(q6_top[
        ["city", "customer_id", "total_spend"]
    ].to_string(index=False))

    assert actual_top_customers == expected_top_customers, (
        "Q6 top customer check failed"
    )

    # Count customers for whom RANK and ROW_NUMBER differ
    rank_difference_count = (
        q6["rank_position"] != q6["row_number_position"]
    ).sum()

    print(
        "\nQ6 rows where RANK differs from ROW_NUMBER:",
        rank_difference_count
    )

    assert rank_difference_count == 8, (
        "Q6 ranking check failed"
    )


    # Q7: Expected 183 daily rows and final cumulative revenue
    q7 = results["Q7"]

    q7_count = len(q7)
    final_cumulative_revenue = q7["cumulative_revenue"].iloc[-1]

    print(f"\nQ7 daily rows: {q7_count}")

    print(
        "Q7 final cumulative revenue: INR",
        f"{final_cumulative_revenue:,.2f}"
    )

    assert q7_count == 183, "Q7 row count check failed"

    assert isclose(
        final_cumulative_revenue,
        2998668.80,
        abs_tol=0.02
    ), "Q7 cumulative revenue check failed"


    # --------------------------------------------------
    # 7. Final validation message
    # --------------------------------------------------

    print("\n" + "=" * 60)
    print("ALL SEVEN REFERENCE CHECKS PASSED!")
    print("=" * 60)


finally:

    # Always close the database connection
    conn.close()

    print("\nDatabase connection closed.")



SQL queries saved successfully: ..\sql\queries.sql

Q1
customer_id  total_spend
      C0693     13776.55
      C0082     12317.70
      C0743     12266.85
      C0525     12241.30
      C0291     12150.70
      C0267     12150.25
      C0488     11825.50
      C0699     11693.90
      C0512     11617.65
      C0332     10779.50
      C0543     10636.10
      C0775     10263.55
      C1024     10160.10
      C0426      9884.60
      C0308      9872.40
      C1147      9740.80
      C0407      9451.75
      C0930      9194.00
      C0130      9170.40
      C0147      9141.50
      C1146      8993.05
      C1136      8983.75
      C0696      8941.75
      C0167      8718.55
      C0819      8695.30
      C0942      8643.60
      C1000      8462.20
      C0916      8395.35
      C0454      8299.15
      C0424      8294.20

Number of rows returned: 30

Q2
     city      category   revenue
Bengaluru         Dairy  87075.05
Bengaluru   Electronics 233909.45
Bengaluru       Grocery 110915.65
B

In [96]:
customers_1 = pd.read_csv('../data/clean/customers_clean.csv')
products_1 = pd.read_csv('../data/clean/products_clean.csv')
orders_1 = pd.read_csv('../data/clean/orders_clean.csv')

print("customers shape:", customers_1.shape)
print("products shape:", products_1.shape)
print("orders shape:", orders_1.shape)

numeric_columns = ["order_amount","quantity","discount_pct","delivery_minutes"]

for column  in numeric_columns:
    orders_1[column] = pd.to_numeric(orders_1[column], errors="coerce")

completed_orders = orders_1[orders_1["status"] == "completed"].copy()

print("Total Completed Orders:", len(completed_orders))

customers shape: (1200, 5)
products shape: (40, 4)
orders shape: (12309, 11)
Total Completed Orders: 11110


In [104]:
print(completed_orders.columns.tolist())

['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes', 'category', 'price_inr']


In [100]:
q2_df = completed_orders.drop(columns=["category"],errors="ignore").merge(
    customers_1[["customer_id", "city"]],on="customer_id", how="inner")

q2_df = q2_df.merge(products_1[["product_id", "category"]], on="product_id", how="inner")

revenue_pivot  = pd.pivot_table(
    q2_df,index= "city", columns="category", values="order_amount", aggfunc="sum", fill_value=0,margins=True,margins_name="All")

print(revenue_pivot.round(2))

pandas_total = revenue_pivot.loc["All", "All"]

print("\nPandas grand total:", round(pandas_total, 2))

category       Dairy  Electronics    Grocery  Personal Care     Snacks  \
city                                                                     
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   
All        531938.95   1219248.10  614293.30      168433.95  464754.50   

category          All  
city                   
Bengaluru   523643.40  
Chennai     463629.15  
Delhi       502829.80  
Hyderabad   528589.90  
Mumbai      479058.50  
Pune        500918.05  
All        2998668.80  

Pandas grand total: 2998668.8


In [101]:
conn = sqlite3.connect("../data/instamart.db")

try:

    sql_result = pd.read_sql_query(
        """
        SELECT SUM(order_amount) AS total_revenue
        FROM orders
        WHERE status = 'completed';
        """,
        conn
    )

    sql_total = sql_result["total_revenue"].iloc[0]

finally:
    conn.close()

print("SQL grand total:", round(sql_total, 2))

SQL grand total: 2998668.8


In [102]:
import numpy as np
print("pandas and sql totals match:", np.isclose(pandas_total, sql_total, atol=0.01))

assert np.isclose(pandas_total, sql_total, atol=0.01), "Pandas and SQL totals do not match!"

assert np.isclose(pandas_total, 2998668.80, atol=0.01), "Pandas total does not match expected value!"

pandas and sql totals match: True


In [105]:
city_orders = completed_orders.merge(
    customers_1[["customer_id", "city"]],
    on="customer_id",
    how="inner"
)

city_summary = (
    city_orders
    .groupby("city")
    .agg(
        total_revenue=("order_amount", "sum"),
        average_order_amount=("order_amount", "mean"),
        order_count=("order_amount", "count")
    )
    .reset_index()
)

print(city_summary.round(2).to_string(index=False))
city_check = city_summary.set_index("city")

assert np.isclose(
    city_check.loc["Bengaluru", "total_revenue"],
    523643.40,
    atol=0.01
)

assert np.isclose(
    city_check.loc["Bengaluru", "average_order_amount"],
    276.47,
    atol=0.01
)

assert city_check.loc["Bengaluru", "order_count"] == 1894

assert np.isclose(
    city_check.loc["Mumbai", "total_revenue"],
    479058.50,
    atol=0.01
)

assert np.isclose(
    city_check.loc["Mumbai", "average_order_amount"],
    297.00,
    atol=0.01
)

assert city_check.loc["Mumbai", "order_count"] == 1613

print("\nCity-wise reference checks passed!")


     city  total_revenue  average_order_amount  order_count
Bengaluru      523643.40                276.47         1894
  Chennai      463629.15                272.72         1700
    Delhi      502829.80                243.62         2064
Hyderabad      528589.90                284.95         1855
   Mumbai      479058.50                297.00         1613
     Pune      500918.05                252.48         1984

City-wise reference checks passed!


EDA

In [106]:
order_amount = completed_orders["order_amount"].dropna()

mean_value = order_amount.mean()
median_value = order_amount.median()
skewness_value = order_amount.skew()

print("Mean:", round(mean_value, 2))
print("Median:", round(median_value, 2))
print("Skewness:", round(skewness_value, 2))

q1 = order_amount.quantile(0.25)
q2 = order_amount.quantile(0.50)
q3 = order_amount.quantile(0.75)

iqr = q3 - q1

print("Q1:", round(q1, 2))
print("Q2 / Median:", round(q2, 2))
print("Q3:", round(q3, 2))
print("IQR:", round(iqr, 2))

Mean: 269.91
Median: 135.0
Skewness: 4.61
Q1: 64.8
Q2 / Median: 135.0
Q3: 297.0
IQR: 232.2


In [107]:
lower_fence = q1 - (1.5 * iqr)
upper_fence = q3 + (1.5 * iqr)

lower_outliers = order_amount[order_amount < lower_fence]

upper_outliers = order_amount[order_amount > upper_fence]

print("Lower fence:", round(lower_fence, 2))
print("Upper fence:", round(upper_fence, 2))

print("Outliers below lower fence:", len(lower_outliers))
print("Outliers above upper fence:", len(upper_outliers))

correlation_columns = ["order_amount", "quantity", "discount_pct", "delivery_minutes"]

correlation_df = completed_orders[correlation_columns]
correlation_matrix = correlation_df.corr()
print(correlation_matrix.round(3))

Lower fence: -283.5
Upper fence: 645.3
Outliers below lower fence: 0
Outliers above upper fence: 897
                  order_amount  quantity  discount_pct  delivery_minutes
order_amount             1.000     0.087        -0.095            -0.003
quantity                 0.087     1.000        -0.000            -0.013
discount_pct            -0.095    -0.000         1.000            -0.010
delivery_minutes        -0.003    -0.013        -0.010             1.000


In [108]:
absolute_corr = correlation_matrix.abs()

upper_triangle = absolute_corr.where(
    np.triu(
        np.ones(absolute_corr.shape),
        k=1
    ).astype(bool)
)

strongest_pair = upper_triangle.stack().idxmax()

strongest_value = correlation_matrix.loc[
    strongest_pair[0],
    strongest_pair[1]
]

print("\n========== STRONGEST CORRELATION ==========")

print("Variable pair:", strongest_pair)
print("Correlation:", round(strongest_value, 3))


========== STRONGEST CORRELATION ==========
Variable pair: ('order_amount', 'discount_pct')
Correlation: -0.095


In [109]:
assert np.isclose(mean_value, 269.91, atol=0.02)
assert np.isclose(median_value, 135.00, atol=0.01)

assert np.isclose(skewness_value, 4.61, atol=0.02)

assert np.isclose(q1, 64.80, atol=0.01)
assert np.isclose(q3, 297.00, atol=0.01)

assert np.isclose(upper_fence, 645.30, atol=0.01)

assert len(lower_outliers) == 0
assert len(upper_outliers) == 897

assert strongest_pair == (
    "order_amount",
    "discount_pct"
)

assert np.isclose(
    strongest_value,
    -0.095,
    atol=0.005
)

print("All reference checks passed!")

All reference checks passed!


In [110]:
summary = f"""
The dataset represents Instamart completed orders, including
order amounts, discounts, quantities and delivery durations.

The mean order amount is INR {mean_value:.2f}, compared with
a median of INR {median_value:.2f}. Skewness is
{skewness_value:.2f}, indicating a strongly right-skewed
distribution.

The IQR is INR {iqr:.2f}, with an upper fence of
INR {upper_fence:.2f}. There are {len(upper_outliers)}
observations above the upper fence and
{len(lower_outliers)} below the lower fence. These high-value
observations should be investigated rather than automatically
removed.

The strongest absolute correlation is between
{strongest_pair[0]} and {strongest_pair[1]}, at
{strongest_value:.3f}. This is a weak negative relationship.
Correlation of this size does not establish causation.
"""

print(summary)

word_count = len(summary.split())

print("Word count:", word_count)

assert word_count <= 200, "Interpretation exceeds 200 words!"


The dataset represents Instamart completed orders, including
order amounts, discounts, quantities and delivery durations.

The mean order amount is INR 269.91, compared with
a median of INR 135.00. Skewness is
4.61, indicating a strongly right-skewed
distribution.

The IQR is INR 232.20, with an upper fence of
INR 645.30. There are 897
observations above the upper fence and
0 below the lower fence. These high-value
observations should be investigated rather than automatically
removed.

The strongest absolute correlation is between
order_amount and discount_pct, at
-0.095. This is a weak negative relationship.
Correlation of this size does not establish causation.

Word count: 97
